In [ ]:
# Ultralytics kütüphanesini Colab ortamına yüklüyoruz
!pip install ultralytics


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 27.6 MB/s eta 0:00:00


In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="qX06jsIdiNDnztKVVG1x")
project = rf.workspace("eminenur").project("ida-wuieq")
version = project.version(1)
dataset = version.download("yolov8")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 130.4 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.18
    Uninstalling idna-3.18:
      Successfully uninstalled idna-3.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
httpx2 2.4.0 requires idna>=3.18, but you have idna 3.7 which i


Extracting Dataset Version Zip to İDA-1 in yolov8:: 100%|██████████| 4443/4443 [00:04<00:00, 1000.77it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
from ultralytics import YOLO

# 1. Önceden eğitilmiş YOLOv8 Nano modelini yüklüyoruz
model = YOLO('yolov8n.pt')

# 2. Eğitimi başlatıyoruz
results = model.train(
    data=f"{dataset.location}/data.yaml",  # İndirilen verinin konumundaki data.yaml dosyasını okur
    epochs=50,                            # İlk deneme (baseline) için 50 epoch çok idealdir
    imgsz=640,                            # Görselleri standart 640x640 boyutuna getirir
    device=0,                             # T4 GPU üzerinde eğitir
    workers=2                             # Colab'de donma olmaması için en kararlı worker sayısı
)

Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/İDA-1/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, pe

In [ ]:
from ultralytics import YOLO
import glob
import os

# En son oluşan best.pt dosyasını bulur
best_pt = sorted(
    glob.glob('/content/runs/detect/train*/weights/best.pt'),
    key=os.path.getmtime
)[-1]

print("Kullanılan model:", best_pt)

model = YOLO(best_pt)
model.export(format='onnx', imgsz=640, opset=12, simplify=True)

In [ ]:
from google.colab import files
import glob
import os

best_onnx = sorted(
    glob.glob('/content/runs/detect/train*/weights/best.onnx'),
    key=os.path.getmtime
)[-1]

files.download(best_onnx)

In [ ]:
from ultralytics import YOLO
import glob
import os
import cv2
from google.colab.patches import cv2_imshow

# 1. En iyi ağırlığımızı yüklüyoruz
model = YOLO('/content/runs/detect/train/weights/best.pt')

# 2. Klasör içindeki resimlerin listesini alıyoruz
test_resimleri = glob.glob('/content/İDA-1/test/images/*.jpg')

# 3. İlk 3 resmi test edip ekrana basacak bir döngü kuralım (hepsi ekrana sığsın diye)
for resim_yolu in test_resimleri[:3]:
    print(f"\n================ RESİM TEST EDİLİYOR: {os.path.basename(resim_yolu)} ================")

    # Model tahmini yapıyor
    results = model.predict(source=resim_yolu, conf=0.50, verbose=False)

    # Koordinat ve renk ayıklama
    for r in results:
        if len(r.boxes) == 0:
            print("Bu karede duba tespit edilemedi.")
        for box in r.boxes:
            x_merkez = int(box.xywh[0][0])
            y_merkez = int(box.xywh[0][1])
            genislik = int(box.xywh[0][2])

            class_id = int(box.cls[0])
            renk_adi = model.names[class_id]
            guven = float(box.conf[0])

            # Arkadaşınızın İDA yönlendirme algoritmasına besleyeceği ham çıktı:
            print(f"📡 [CANLI DATA] -> Renk: {renk_adi:<7} | Merkez X: {x_merkez:<3} | Merkez Y: {y_merkez:<3} | Genişlik: {genislik}px")

    # Görseli çizdirip gösterelim
    cizimli_gorsel = results[0].plot()
    cv2_imshow(cizimli_gorsel)
    print(model.names)
# Örnek çıktı: {0: 'black', 1: 'green', 2: 'orange', 3: 'red', 4: 'yellow'}
    print("========================================================================\n")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


FileNotFoundError: [Errno 2] No such file or directory: '/content/runs/detect/train/weights/best.pt'

In [ ]:
pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.7 MB/s eta 0:00:00
